# चुनौती: डेटा विज्ञान के बारे में पाठ का विश्लेषण

इस उदाहरण में, चलिए एक सरल अभ्यास करते हैं जो पारंपरिक डेटा विज्ञान प्रक्रिया के सभी चरणों को कवर करता है। आपको कोई कोड लिखने की आवश्यकता नहीं है, आप नीचे सेल्स पर क्लिक करके उन्हें निष्पादित कर सकते हैं और परिणाम देख सकते हैं। एक चुनौती के रूप में, आपको प्रोत्साहित किया जाता है कि आप इस कोड को अलग-अलग डेटा के साथ आजमाएं।

## लक्ष्य

इस पाठ में, हमने डेटा विज्ञान से संबंधित विभिन्न अवधारणाओं की चर्चा की है। चलिए कुछ **पाठ खनन (text mining)** करके और संबंधित अवधारणाओं को खोजने की कोशिश करते हैं। हम डेटा विज्ञान के बारे में एक पाठ से शुरू करेंगे, उसमें से प्रमुख शब्द निकालेंगे, और फिर परिणाम को दृश्य रूप में प्रस्तुत करने का प्रयास करेंगे।

एक पाठ के रूप में, मैं विकिपीडिया से डेटा विज्ञान के पृष्ठ का उपयोग करूंगा:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## चरण 1: डेटा प्राप्त करना

हर डेटा साइंस प्रक्रिया का पहला कदम डेटा प्राप्त करना होता है। हम इसे करने के लिए `requests` लाइब्रेरी का उपयोग करेंगे:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## चरण 2: डेटा को रूपांतरित करना

अगला चरण डेटा को उस रूप में परिवर्तित करना है जो प्रसंस्करण के लिए उपयुक्त हो। हमारे मामले में, हमने पृष्ठ से HTML स्रोत कोड डाउनलोड किया है, और हमें इसे साधारण पाठ में परिवर्तित करना है।

इसे करने के कई तरीके हैं। हम [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) का उपयोग करेंगे, जो HTML पार्सिंग के लिए एक लोकप्रिय Python पुस्तकालय है। BeautifulSoup हमें विशिष्ट HTML तत्वों को लक्षित करने की अनुमति देता है, ताकि हम विकिपीडिया के मुख्य लेख सामग्री पर ध्यान केंद्रित कर सकें और कुछ नेविगेशन मेनू, साइडबार, फुटर और अन्य अप्रासंगिक सामग्री को कम कर सकें (हालांकि कुछ बुनियादी टेक्स्ट अभी भी रह सकता है)।


सबसे पहले, हमें HTML पार्सिंग के लिए BeautifulSoup लाइब्रेरी इंस्टॉल करनी होगी:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## चरण 3: अंतर्दृष्टि प्राप्त करना

सबसे महत्वपूर्ण चरण हमारे डेटा को कुछ ऐसे रूप में बदलना है जिससे हम अंतर्दृष्टि निकाल सकें। हमारे मामले में, हम टेक्स्ट से कीवर्ड निकालना चाहते हैं, और देखना चाहते हैं कि कौन से कीवर्ड अधिक महत्वपूर्ण हैं।

हम कीवर्ड एक्सट्रैक्शन के लिए Python लाइब्रेरी [RAKE](https://github.com/aneesha/RAKE) का उपयोग करेंगे। पहले, अगर यह लाइब्रेरी मौजूद नहीं है तो इसे इंस्टॉल कर लेते हैं: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

मुख्य कार्यक्षमता `Rake` ऑब्जेक्ट से उपलब्ध है, जिसे हम कुछ पैरामीटर का उपयोग करके अनुकूलित कर सकते हैं। हमारे मामले में, हम एक कीवर्ड की न्यूनतम लंबाई 5 वर्ण, दस्तावेज़ में एक कीवर्ड की न्यूनतम आवृत्ति 3, और एक कीवर्ड में शब्दों की अधिकतम संख्या 2 सेट करेंगे। अन्य मानों के साथ खेलने के लिए स्वतंत्र महसूस करें और परिणाम का निरीक्षण करें।


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


हमने महत्व के दर्जे के साथ शर्तों की एक सूची प्राप्त की। जैसा कि आप देख सकते हैं, सबसे प्रासंगिक विषय, जैसे कि मशीन लर्निंग और बिग डेटा, सूची में शीर्ष स्थानों पर मौजूद हैं।

## चरण 4: परिणाम को दृश्य रूप में प्रस्तुत करना

लोग डेटा को सबसे अच्छी तरह से दृश्य रूप में समझ सकते हैं। इसलिए अक्सर कुछ अंतर्दृष्टि प्राप्त करने के लिए डेटा को दृश्य रूप में प्रस्तुत करना समझदारी होती है। हम `matplotlib` पुस्तकालय का उपयोग करके Python में कीवर्ड्स के प्रासंगिकता के साथ सरल वितरण का प्लॉट बना सकते हैं:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

हालांकि, शब्द आवृत्तियों को देखने का एक और भी बेहतर तरीका है - **Word Cloud** का उपयोग करना। हम अपनी कीवर्ड सूची से वर्ड क्लाउड बनाने के लिए एक अन्य लाइब्रेरी इंस्टॉल करने की आवश्यकता होगी।


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` ऑब्जेक्ट या तो मूल टेक्स्ट लेता है, या पूर्व-गणना किए गए शब्दों की आवृत्तियों वाली सूची लेता है, और एक छवि लौटाता है, जिसे फिर `matplotlib` का उपयोग करके प्रदर्शित किया जा सकता है:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

हम मूल पाठ को `WordCloud` में भी पास कर सकते हैं - देखते हैं कि क्या हम समान परिणाम प्राप्त कर पाते हैं:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

आप देख सकते हैं कि वर्ड क्लाउड अब अधिक प्रभावशाली दिखता है, लेकिन इसमें बहुत अधिक शोर भी है (जैसे, अप्रासंगिक शब्द जैसे `Retrieved on`)। साथ ही, हमें कम कीवर्ड मिलते हैं जो दो शब्दों से बने होते हैं, जैसे *data scientist*, या *computer science*। इसका कारण यह है कि RAKE एल्गोरिथ्म टेक्स्ट से अच्छे कीवर्ड चुनने में बेहतर काम करता है। यह उदाहरण डेटा प्री-प्रोसेसिंग और सफाई के महत्व को दर्शाता है, क्योंकि अंत में स्पष्ट तस्वीर हमें बेहतर निर्णय लेने में सक्षम बनाएगी।

इस अभ्यास में हमने विकिपीडिया टेक्स्ट से कुछ अर्थ निकालने की एक सरल प्रक्रिया देखी, कीवर्ड और वर्ड क्लाउड के रूप में। यह उदाहरण काफी सरल है, लेकिन यह उन सभी सामान्य चरणों को अच्छी तरह से प्रदर्शित करता है जो एक डेटा वैज्ञानिक डेटा पर काम करते समय अपनाता है, डेटा अधिग्रहण से शुरू होकर, विज़ुअलाइज़ेशन तक।

हमारे कोर्स में हम उन सभी चरणों पर विस्तार से चर्चा करेंगे।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
